In [ ]:

# IMPORT LIBRARIES

import pandas as pd
import os

# CHANGE ONLY THIS LINE

FILE_NAME = "Uplaod File Path Here"

# Examples:
# BTCUSDT_1m_2018_2024.csv
# BTCUSDT_5m_2018_2024.csv
# BTCUSDT_15m_2018_2024.csv
# BTCUSDT_30m_2018_2024.csv
# BTCUSDT_1h_2018_2024.csv


# INPUT & OUTPUT PATHS

INPUT_PATH = f"/content/{FILE_NAME}"

OUTPUT_DIR = "/content/drive/MyDrive/Preprocessed_Binance_Datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# LOAD DATASET

print(f"Loading Dataset: {FILE_NAME}")

df = pd.read_csv(INPUT_PATH, header=None, low_memory=False)


# ASSIGN COLUMN NAMES

df.columns = [
    "Open_Time",
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Close_Time",
    "Quote_Asset_Volume",
    "Number_of_Trades",
    "Taker_Buy_Base_Volume",
    "Taker_Buy_Quote_Volume",
    "Ignore"
]


# CONVERT NUMERIC COLUMNS

numeric_columns = [
    "Open_Time",
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Close_Time",
    "Quote_Asset_Volume",
    "Number_of_Trades",
    "Taker_Buy_Base_Volume",
    "Taker_Buy_Quote_Volume",
    "Ignore"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# RECORD INITIAL ROWS

rows_before = len(df)


# REMOVE MISSING VALUES

missing_before = df.isnull().sum().sum()

df.dropna(inplace=True)


# REMOVE DUPLICATES

duplicates_removed = df.duplicated().sum()

df.drop_duplicates(inplace=True)


# CONVERT TIMESTAMPS

df["Open_Time"] = pd.to_datetime(df["Open_Time"], unit="ms", errors='coerce')
df["Close_Time"] = pd.to_datetime(df["Close_Time"], unit="ms", errors='coerce')
# Remove rows where timestamp conversion resulted in NaT
df.dropna(subset=["Open_Time", "Close_Time"], inplace=True)


# SORT DATASET

df.sort_values("Open_Time", inplace=True)
df.reset_index(drop=True, inplace=True)

# SAVE PREPROCESSED DATASET

output_file = os.path.join(
    OUTPUT_DIR,
    FILE_NAME.replace(".csv", "_Preprocessed.csv")
)

df.to_csv(output_file, index=False)


# PREPROCESSING SUMMARY

print("\n" + "="*70)
print("PREPROCESSING SUMMARY")
print("="*70)

print(f"Dataset               : {FILE_NAME}")
print(f"Study Period          : {df['Open_Time'].min()}  -->  {df['Open_Time'].max()}")
print(f"Rows Before Cleaning  : {rows_before:,}")
print(f"Rows After Cleaning   : {len(df):,}")
print(f"Columns               : {df.shape[1]}")
print(f"Missing Values Removed: {missing_before:,}")
print(f"Duplicate Rows Removed: {duplicates_removed:,}")

print("\nMissing Values Per Column")
print(df.isnull().sum())

print("\nData Types")
print(df.dtypes)

print("\nFirst 5 Rows")
print(df.head())

print("\nLast 5 Rows")
print(df.tail())

print("\n" + "="*70)
print("PREPROCESSING COMPLETED SUCCESSFULLY")
print("="*70)
print(f"Preprocessed Dataset Saved To:\n{output_file}")